# Public Complaint Analysis

## Project Overview
Analyze public grievance or complaint records to identify recurring issue categories, agencies involved, and resolution delays.

## Learning Objectives
- Categorize and rank complaint types by volume and severity
- Analyze resolution time distributions across agencies and categories
- Identify seasonal or geographic patterns in complaints
- Quantify agency responsiveness and backlog trends

## Problem Statement
City or government officials need to understand which complaint categories are most common, which agencies are slowest to respond, and where systemic service gaps exist.

## Why This Project Matters
Public complaint data is a direct measure of citizen experience. Analyzing it drives accountability, resource allocation, and service improvement.

## Dataset Overview
Synthetic complaint dataset: ~8,000 records with category, agency, location, timestamps, and resolution data — inspired by NYC 311 data.

## Dataset Source and License Notes
- Synthetic data inspired by NYC 311 / Open Data portals
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 8000
categories = np.random.choice(['Noise', 'Pothole', 'Illegal Parking', 'Street Light Out', 'Trash Collection',
                                 'Water Leak', 'Graffiti', 'Building Violation', 'Rodent Sighting', 'Other'],
                                n, p=[0.15, 0.13, 0.12, 0.10, 0.12, 0.08, 0.07, 0.08, 0.08, 0.07])
agencies = np.random.choice(['DOT', 'Sanitation', 'Police', 'Housing', 'DEP', 'Parks', 'Buildings'],
                              n, p=[0.20, 0.18, 0.18, 0.12, 0.12, 0.10, 0.10])
boroughs = np.random.choice(['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'],
                              n, p=[0.25, 0.25, 0.22, 0.18, 0.10])
created = pd.Timestamp('2023-01-01') + pd.to_timedelta(np.random.randint(0, 730, n), unit='D')
resolution_days = np.clip(np.random.lognormal(2.0, 0.9, n), 0.1, 180).round(1)
status = np.where(resolution_days < 30, 'Closed',
         np.where(resolution_days < 90, np.random.choice(['Closed', 'In Progress'], n), 'Open'))
priority = np.random.choice(['Low', 'Medium', 'High', 'Emergency'], n, p=[0.30, 0.35, 0.25, 0.10])

df = pd.DataFrame({
    'ComplaintID': [f'C{i:06d}' for i in range(n)],
    'Category': categories, 'Agency': agencies, 'Borough': boroughs,
    'CreatedDate': created, 'ResolutionDays': resolution_days,
    'Status': status, 'Priority': priority,
})
df['Month'] = df['CreatedDate'].dt.to_period('M')
df['DayOfWeek'] = df['CreatedDate'].dt.day_name()
df['Year'] = df['CreatedDate'].dt.year

print(f'Dataset: {df.shape}')
print(f'Date range: {df["CreatedDate"].min().date()} to {df["CreatedDate"].max().date()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nCategory distribution:\n{df["Category"].value_counts()}')
print(f'\nStatus distribution: {df["Status"].value_counts().to_dict()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df['Category'].value_counts().plot.barh(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Complaints by Category')
df.groupby('Agency')['ComplaintID'].count().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Complaints by Agency')
df.groupby('Borough')['ComplaintID'].count().sort_values().plot.barh(ax=axes[1,0], edgecolor='black', color='teal')
axes[1,0].set_title('Complaints by Borough')
monthly = df.groupby(df['CreatedDate'].dt.to_period('M'))['ComplaintID'].count()
monthly.index = monthly.index.to_timestamp()
monthly.plot(ax=axes[1,1], marker='.', color='goldenrod')
axes[1,1].set_title('Monthly Complaint Volume')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Resolution Time Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
agency_res = df.groupby('Agency')['ResolutionDays'].median().sort_values()
agency_res.plot.barh(ax=axes[0], edgecolor='black', color='coral')
axes[0].set_title('Median Resolution Days by Agency')
cat_res = df.groupby('Category')['ResolutionDays'].median().sort_values()
cat_res.plot.barh(ax=axes[1], edgecolor='black', color='steelblue')
axes[1].set_title('Median Resolution Days by Category')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'resolution_time.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Resolution time stats by Agency:')
print(df.groupby('Agency')['ResolutionDays'].agg(['median','mean','std','count']).round(1).sort_values('median', ascending=False))

## Priority and Backlog Analysis

In [ ]:
priority_stats = df.groupby('Priority').agg(
    complaints=('ComplaintID', 'count'),
    median_resolution=('ResolutionDays', 'median'),
    pct_open=('Status', lambda x: (x == 'Open').mean()),
).round(3)
print('Priority Analysis:')
print(priority_stats)

backlog = df[df['Status'] != 'Closed']
print(f'\nOpen/In-Progress backlog: {len(backlog)} ({len(backlog)/len(df):.1%})')
print(f'Backlog by Agency:\n{backlog["Agency"].value_counts()}')

## Day-of-Week and Seasonal Patterns

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
fig, ax = plt.subplots(figsize=(10, 5))
df.groupby('DayOfWeek')['ComplaintID'].count().reindex(dow_order).plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Complaints by Day of Week')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'day_of_week.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Noise and pothole complaints** dominate volume — these are quality-of-life issues with high citizen visibility
- **Resolution times vary significantly** across agencies — some agencies consistently take 2-3x longer than others
- **High-priority complaints** should resolve faster, but the data shows moderate variation — priority routing may need improvement
- **Backlog** is concentrated in specific agencies — staffing or process bottlenecks exist
- **Seasonal patterns** may emerge (more potholes after winter, more noise in summer)
- **Borough-level analysis** reveals unequal service delivery

## Limitations
- Synthetic data with simplified resolution logic
- No text analysis of complaint descriptions
- No citizen satisfaction or follow-up data
- No cost or resource allocation data
- No geospatial analysis beyond borough level

## How to Improve This Project
- Add NLP analysis of complaint text for sub-categorization
- Include geospatial mapping of complaint hotspots
- Track repeat complaints at same location
- Add citizen satisfaction surveys post-resolution
- Build SLA compliance dashboards by agency

## Production Considerations
- Real-time complaint volume dashboards for city operations
- Automated SLA breach alerts by agency
- Predictive models for complaint volume forecasting
- Public-facing transparency portals showing resolution metrics

## Common Mistakes
- Comparing agencies without normalizing for complaint complexity
- Using mean instead of median for resolution time (highly skewed)
- Not tracking reopened or repeat complaints
- Ignoring seasonal baselines when evaluating performance

## Mini Challenge / Exercises
1. Calculate the % of Emergency priority complaints resolved within 48 hours.
2. Which Borough × Category combination has the longest median resolution?
3. Estimate the staffing impact if the target was to resolve all complaints within 14 days.
4. Build a simple agency performance scorecard using volume, resolution time, and backlog %.

## Final Summary / Key Takeaways
- Public complaint analysis provides direct insight into government service quality
- Category and agency segmentation reveals where performance gaps exist
- Resolution time tracking is the key accountability metric
- Backlog management and priority routing need continuous monitoring
- Data transparency builds public trust and drives service improvement